# Family fetch: collect every protein that carries the PDC-family domains

This notebook collects, for the taxa you choose, **all UniProt proteins that InterPro or Pfam annotate with any of a list of domain entries** (lipoyl / biotin-lipoyl attachment, E3-binding (peripheral subunit-binding), 2-oxoacid acyltransferase catalytic domain, and related family entries). It takes the **union** of all those queries, records where each domain lies in each protein, labels every protein with a **domain architecture**, writes FASTA and table files, and can build a small standalone **MMseqs2 database** that notebooks B and C accept as `DATABASE`.

## What it does, step by step
| Step | What happens | Typical cost | Output |
|---|---|---|---|
| 1 Environment | Finds MMseqs2 (only if a database is built) and defines helpers | Seconds | Versions, folders |
| 2 Query list | Cleans the InterPro and Pfam ID lists to unique IDs, shows what each ID is and which are broad | Instant | Table of IDs × taxa |
| 3 Counts | Asks InterPro how many proteins each ID × taxon has (one small request each) | A few seconds per query (not measured) | Count table; stop here with `COUNT_ONLY` |
| 4 Fetch and union | Downloads all proteins for each ID × taxon with domain positions and sequences, caches each query, merges by UniProt accession | About one request per 200 proteins plus a pause; not measured | One row per protein |
| 5 Architecture | Merges domain positions into lipoyl (L), PSBD (P) and catalytic (C) regions, writes an architecture string and a preliminary class | Seconds | Class table |
| 6 Reference check | Optional: compare the union with a FASTA you supply (accession, identical or similar sequence), Venn diagram, list of missing ones, why they were missed, option to add them | Seconds; similarity search seconds to a minute | Venn, tables |
| 7 Output | `proteins.fasta`, `proteins.tsv`, `domains.tsv`, one FASTA of regions per domain type | Seconds | Files in `OUT_DIR` |
| 8 Database | Optional: builds a standalone MMseqs2 database with taxonomy (same format as notebook A) | About a million sequences per minute for conversion (earlier Colab run), plus a taxonomy download | Folder + `.tar` |

## How the union is done
Each ID is queried separately and each answer lists the proteins that carry that entry. The notebook merges the answers **by UniProt accession in Python**: a protein found through three IDs becomes one row that lists all three IDs and all their domain positions. This is the simplest way and keeps the domain positions, which a single combined "A OR B OR C" text query to UniProt would not give.

## Things to keep in mind
- **Annotation-based.** A protein is found only if InterPro or Pfam matches it above their thresholds. Diverged members below those thresholds are missed, and so are genes missing from the UniProt annotation. Treat this set as a **seed set and a lower bound**, not as "everything". Notebook B/C searches and your own profile searches are what reach further.
- **Some IDs are broad.** Entries named as superfamilies or generic folds (for example "single hybrid motif", "CAT-like domain superfamily") can match many unrelated proteins, such as biotin carriers or other acyltransferases. Check the count table and remove IDs with `EXCLUDE_IDS` if one dominates.
- **The architecture class is a heuristic label** from the annotation, not a functional assignment. Use notebook C for the E2 versus E3BP decision.
- **Taxon ids include all descendants.** The InterPro API takes one taxid at a time; fungi (4751) covers the whole kingdom.
- **Internet access is required** for steps 3 to 4 (InterPro API, EBI) and for the taxonomy download in step 7. On a cluster this normally means a login node or a node with outbound access.
- **Not verified against the live API.** The notebook was written from the request format in your draft cells and was tested here only against a local mock of the API. The first run on the real service may need small fixes; errors name the URL that failed.

## Deep alignments, if you cluster this set
Notes are at the end of the notebook (step 9).

In [ ]:
#@title Parameters { display-mode: "form" }
#@markdown ### Taxa
#@markdown Tick the groups. Fungi, Metazoa (animals), Viridiplantae (plants) and Bacteria are NCBI taxids 4751, 33208, 33090 and 2. Add others as comma-separated taxids. Each taxon includes everything below it.
TAX_FUNGI = True #@param {type:"boolean"}
TAX_METAZOA = False #@param {type:"boolean"}
TAX_PLANTS = False #@param {type:"boolean"}
TAX_BACTERIA = False #@param {type:"boolean"}
EXTRA_TAXIDS = "" #@param {type:"string"}
#@markdown ### Domain entries (the union of all of these is collected)
#@markdown Paste IDs separated by commas, spaces or new lines; names glued to the IDs, duplicates and lower case are cleaned up. InterPro IDs look like `IPR004167`, Pfam IDs like `PF02817`. `EXCLUDE_IDS` drops IDs from the lists without editing them.
INTERPRO_IDS = "IPR000089, IPR001078, IPR003016, IPR004167, IPR006255, IPR006257, IPR011053, IPR023213, IPR036625, IPR045257, IPR050537" #@param {type:"string"}
PFAM_IDS = "PF00198, PF00364, PF02817" #@param {type:"string"}
SIGNATURE_IDS = "G3DSA:4.10.320.10, SSF47005" #@param {type:"string"}
#@markdown `SIGNATURE_IDS`: member-database signatures that InterPro has not always merged into an InterPro entry: Gene3D (`G3DSA:4.10.320.10`), SUPERFAMILY (`SSF47005`), PANTHER family or subfamily (`PTHR23151`, `PTHR23151:SF82`). They are queried like the other IDs and join the union.
EXCLUDE_IDS = "" #@param {type:"string"}
#@markdown ### Fetch
#@markdown `FETCH_BROAD`: entries with role `broad` (generic superfamily entries, see Advanced cell) are only counted unless this is ticked; fetching them can add tens of thousands of unrelated proteins.
FETCH_BROAD = False #@param {type:"boolean"}
#@markdown `COUNT_ONLY` stops after the count table. `REVIEWED_ONLY` keeps only Swiss-Prot (reviewed) entries. `MIN_LENGTH` drops shorter proteins (0 = keep all). `REFETCH` ignores cached answers.
COUNT_ONLY = False #@param {type:"boolean"}
REVIEWED_ONLY = False #@param {type:"boolean"}
MIN_LENGTH = 0 #@param {type:"integer"}
REFETCH = False #@param {type:"boolean"}
#@markdown `SEQUENCES_FROM_UNIPROT` (recommended): the InterPro pages are fetched without sequences, which is much faster; the sequences (and any missing taxids) are then downloaded from UniProt in batches of 200 accessions. Untick to get sequences from InterPro in the same pages (slow).
SEQUENCES_FROM_UNIPROT = True #@param {type:"boolean"}
#@markdown ### Inspection files
#@markdown `WRITE_ARCH_FILES`: after step 5, write one FASTA per class and architecture to `OUT_DIR/architectures/`, plus a summary table. Architectures with fewer than `ARCH_MIN_SIZE` proteins are pooled into `rare.fasta` (the summary still lists them).
WRITE_ARCH_FILES = True #@param {type:"boolean"}
ARCH_MIN_SIZE = 5 #@param {type:"integer"}
PAUSE_SECONDS = 1.0 #@param {type:"number"}
VERIFY_SSL = True #@param {type:"boolean"}
#@markdown ### Check against a reference set (optional)
#@markdown `REFERENCE_FASTA`: path to a FASTA of sequences that should be in the union (empty = skip). Sequences are matched by UniProt accession, identical sequence, or (if `MATCH_SIMILAR`) by similarity at `MATCH_IDENTITY` and `MATCH_COVERAGE`. `INCLUDE_MISSING` adds the unmatched ones to the union. `DIAGNOSE_MISSING` asks UniProt why up to 50 of them were missed.
REFERENCE_FASTA = "" #@param {type:"string"}
UPLOAD_REFERENCE = False #@param {type:"boolean"}
MATCH_SIMILAR = True #@param {type:"boolean"}
MATCH_IDENTITY = 0.95 #@param {type:"number"}
MATCH_COVERAGE = 0.8 #@param {type:"number"}
INCLUDE_MISSING = False #@param {type:"boolean"}
DIAGNOSE_MISSING = True #@param {type:"boolean"}
#@markdown ### Output and database
OUT_DIR = "" #@param {type:"string"}
WORKDIR = "" #@param {type:"string"}
BUILD_DB = True #@param {type:"boolean"}
DB_NAME = "family_fungi" #@param {type:"string"}
LIBRARY_DIR = "" #@param {type:"string"}
SELF_TEST = True #@param {type:"boolean"}
PACKAGE_TAR = True #@param {type:"boolean"}
ARCHIVE_DIR = "" #@param {type:"string"}
DOWNLOAD_PACKAGE = False #@param {type:"boolean"}
OVERWRITE_EXISTING = False #@param {type:"boolean"}
#@markdown ### Tools (only needed to build the database)
MMSEQS_EXE = "" #@param {type:"string"}
AUTO_INSTALL = True #@param {type:"boolean"}

In [ ]:
# Advanced (optional).
# Folder with an NCBI taxdump (names.dmp, nodes.dmp, merged.dmp) for the database taxonomy. Empty = download from NCBI.
NCBI_TAXDUMP = ""
# Paste a reference FASTA here instead of using a file (used only when REFERENCE_FASTA is empty and nothing is uploaded).
REFERENCE_INLINE = """"""
# Service addresses (change only for a mirror or for testing).
API_BASE = "https://www.ebi.ac.uk/interpro/api"
UNIPROT_BASE = "https://rest.uniprot.org"
PAGE_SIZE = 200  # the InterPro server returns at most 200 proteins per page whatever is asked; a larger value only breaks the time estimates
# Role of each entry in the architecture string. L = lipoyl / biotin-lipoyl attachment, P = peripheral subunit-binding (E3-binding),
# C = acyltransferase catalytic domain. Entries with another role are listed in the table but do not enter the architecture.
# "broad" marks generic superfamily-level entries that can match unrelated proteins.
ENTRY_ROLE = {
    "IPR000089": "L", "PF00364": "L", "IPR003016": "L",
    "IPR004167": "P", "PF02817": "P", "IPR036625": "P",
    "IPR001078": "C", "PF00198": "C",
    "IPR011053": "broad", "IPR023213": "broad",
    "G3DSA:4.10.320.10": "P", "SSF47005": "P", "PTHR23151:SF82": "family",
    "IPR045257": "family", "IPR006255": "family", "IPR006257": "family", "IPR050537": "family",
}
# Names as given in your list (for display only).
ENTRY_NAME = {
    "IPR000089": "Biotin_lipoyl", "IPR001078": "2-oxoacid_DH_actylTfrase", "IPR003016": "2-oxoA_DH_lipoyl-BS",
    "IPR004167": "PSBD", "IPR006255": "SucB", "IPR006257": "LAT1", "IPR011053": "Single_hybrid_motif",
    "IPR023213": "CAT-like_dom_sf", "IPR036625": "E3-binding / E3-bd_dom_sf", "IPR045257": "E2/Pdx1",
    "IPR050537": "2-oxoacid_dehydrogenase", "PF00198": "2-oxoacid_dh", "PF00364": "Biotin_lipoyl", "PF02817": "E3_binding",
}

## 1. Environment and helpers
**What happens.** Defines the helpers used below: a program runner without a shell (so it behaves the same on Linux, macOS and Windows), a resumable downloader, the database builder, the check-and-package routine. MMseqs2 is looked for (`MMSEQS_EXE`, then the PATH, then an automatic install on Linux/macOS) only if `BUILD_DB` is ticked.

**Expect.** Seconds. Output: system, Python and MMseqs2 versions, folders.

In [ ]:
#@title 1. Environment and helpers { display-mode: "form" }
import os, sys, re, json, shutil, hashlib, subprocess, time, platform, gzip, datetime, tarfile, zipfile, urllib.parse, urllib.request, urllib.error
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SYSTEM = platform.system()
THREADS = os.cpu_count() or 2
_base = Path("/content") if IN_COLAB else Path.cwd()
WORK = Path(WORKDIR) if WORKDIR else _base / "work_dbbuild"
WORK.mkdir(parents=True, exist_ok=True)
TMP = WORK / "tmp"; TMP.mkdir(exist_ok=True)
LIB = Path(LIBRARY_DIR) if LIBRARY_DIR else _base / "mmseqs_library"
OUT = Path(OUT_DIR) if OUT_DIR else WORK / "family_fetch_results"
LIB.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
BIN = WORK / "bin"
NB_VERSION = "F1"
PROGRESS_RE = re.compile(r"^\s*\[[=> ]*\]")

def run(args, quiet=False, tail=25, every=20):
    """Run a program (list of arguments, no shell). MMseqs2 progress-bar lines are shown at most once per `every` seconds."""
    args = [str(a) for a in args]
    if not quiet:
        print("$", " ".join(f'"{a}"' if " " in a else a for a in args)[:300])
    t0, last, lines = time.time(), 0.0, []
    try:
        p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    except OSError as e:
        raise RuntimeError(f"Could not start {args[0]!r}: {e}")
    for line in p.stdout:
        lines.append(line)
        if quiet:
            continue
        if PROGRESS_RE.match(line):
            if time.time() - last >= every:
                print(f"  [{time.time() - t0:6.0f}s] {line.strip()[:110]}"); last = time.time()
        else:
            print(line, end="")
    rc = p.wait()
    if rc:
        raise RuntimeError(f"Command failed (exit {rc}): {' '.join(args)[:300]}\n" + "".join(lines[-tail:]))
    if not quiet:
        print(f"  (finished in {time.time() - t0:.0f}s)")
    return "".join(lines)

def _find(names, extra_dirs=()):
    for n in names:
        w = shutil.which(n)
        if w: return w
    for d in extra_dirs:
        if Path(d).exists():
            for n in names:
                for p in Path(d).rglob(n):
                    if p.is_file(): return str(p)
    return None

def find_mmseqs():
    if MMSEQS_EXE:
        if not Path(MMSEQS_EXE).exists(): raise FileNotFoundError(f"MMSEQS_EXE {MMSEQS_EXE!r} does not exist.")
        return MMSEQS_EXE
    return _find(["mmseqs", "mmseqs.bat", "mmseqs.exe"], [BIN])

def install_mmseqs():
    BIN.mkdir(parents=True, exist_ok=True)
    base = "https://github.com/soedinglab/MMseqs2/releases/latest/download/"
    if SYSTEM == "Linux":
        if platform.machine().lower() in ("aarch64", "arm64"):
            asset = "mmseqs-linux-arm64.tar.gz"
        else:
            try: flags = open("/proc/cpuinfo").read()
            except OSError: flags = ""
            asset = "mmseqs-linux-" + ("avx2" if " avx2" in flags else ("sse41" if " sse4_1" in flags else "sse2")) + ".tar.gz"
    elif SYSTEM == "Darwin":
        asset = "mmseqs-osx-universal.tar.gz"
    else:
        raise RuntimeError("There is no automatic MMseqs2 install for this system. On Windows use WSL2 (see the notes at the top of this notebook), "
                           "or install MMseqs2 yourself and set MMSEQS_EXE.")
    dest = BIN / asset
    print("Downloading", base + asset)
    urllib.request.urlretrieve(base + asset, dest)
    with tarfile.open(dest) as t: t.extractall(BIN)
    dest.unlink()

if BUILD_DB:
    MMSEQS = find_mmseqs()
    if not MMSEQS and AUTO_INSTALL:
        install_mmseqs(); MMSEQS = find_mmseqs()
    if not MMSEQS:
        raise RuntimeError("MMseqs2 not found. Install it (conda: `conda install -c conda-forge -c bioconda mmseqs2`) and set MMSEQS_EXE if it is not on the PATH, or untick BUILD_DB.")
else:
    MMSEQS = None

def mm(*args, **kw):
    return run([MMSEQS, *args], **kw)

MMSEQS_VERSION = mm("version", quiet=True).strip().splitlines()[-1] if MMSEQS else "not used"

def download(url, dest, resume=False, every=20):
    """Download with a progress line every `every` seconds. With resume=True a partial file is continued if the server allows it."""
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    done = dest.stat().st_size if (resume and dest.exists()) else 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={done}-"} if done else {})
    try:
        r = urllib.request.urlopen(req, timeout=60)
    except urllib.error.HTTPError as e:
        if e.code == 416 and done:             # already complete
            return dest
        raise RuntimeError(f"Download failed ({e.code}) for {url}")
    total = r.headers.get("Content-Length"); total = int(total) + done if total else None
    if done and r.status != 206:
        done = 0                                # server ignored the range: start over
    mode = "ab" if done else "wb"
    t0 = last = time.time(); n = done
    with open(dest, mode) as f:
        while True:
            b = r.read(1 << 20)
            if not b: break
            f.write(b); n += len(b)
            if time.time() - last >= every:
                print(f"  {n / 1e9:.2f}" + (f" / {total / 1e9:.2f}" if total else "") + f" GB after {time.time() - t0:.0f}s"); last = time.time()
    if total and n != total:
        raise RuntimeError(f"Download incomplete: {n} of {total} bytes. Run the cell again to resume.")
    print(f"  downloaded {n / 1e9:.2f} GB in {time.time() - t0:.0f}s -> {dest}")
    return dest

def keyof(*objs):
    return hashlib.sha1(json.dumps(objs, sort_keys=True, default=str).encode()).hexdigest()[:12]

def n_seqs(db):
    return sum(1 for _ in open(str(db) + ".index"))

def dir_size_gb(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1e9

def ensure_taxdump():
    """Folder with names.dmp, nodes.dmp, merged.dmp: NCBI_TAXDUMP if given, else downloaded from NCBI once."""
    if NCBI_TAXDUMP:
        d = Path(NCBI_TAXDUMP)
        if not (d / "nodes.dmp").exists(): raise FileNotFoundError(f"No nodes.dmp in NCBI_TAXDUMP {d}")
        return d
    d = WORK / "taxdump"
    if (d / "nodes.dmp").exists() and (d / "names.dmp").exists():
        return d
    d.mkdir(parents=True, exist_ok=True)
    tgz = download("https://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz", WORK / "taxdump.tar.gz")
    with tarfile.open(tgz) as t:
        for m in t.getmembers():
            if m.name in ("names.dmp", "nodes.dmp", "merged.dmp", "delnodes.dmp"):
                m.name = Path(m.name).name; t.extract(m, d)
    tgz.unlink()
    return d

def descendants(taxids, dump):
    """All taxids at or below the given ones (from nodes.dmp), plus old taxids merged into them (merged.dmp)."""
    children = {}
    with open(Path(dump) / "nodes.dmp") as f:
        for line in f:
            p = line.split("\t|\t")
            t, par = int(p[0]), int(p[1])
            if t != par: children.setdefault(par, []).append(t)
    out, stack = set(), list(taxids)
    while stack:
        t = stack.pop()
        if t in out: continue
        out.add(t); stack.extend(children.get(t, ()))
    mf = Path(dump) / "merged.dmp"
    if mf.exists():
        with open(mf) as f:
            for line in f:
                p = line.split("\t|\t"); old, new = int(p[0]), int(p[1].split("\t")[0])
                if new in out: out.add(old)
    return out

def build_db_from_fasta(fa, db, workdir):
    """FASTA (plain or gzip) -> MMseqs2 database with taxonomy. Headers must carry OX=taxid or TaxID=taxid."""
    db, workdir = Path(db), Path(workdir)
    mm("createdb", fa, db, "-v", 2)
    with open(fa, "rb") as fh:
        is_gz = fh.read(2) == b"\x1f\x8b"
    mapf = workdir / "taxmap.tsv"; n_h = n_miss = 0
    # createdb stores the sequence identifiers in db.lookup in input order (for sp|ACC|NAME headers it keeps only ACC),
    # so the mapping is written in lockstep with the headers, using the lookup's own identifier.
    with (gzip.open if is_gz else open)(fa, "rt") as fi, open(mapf, "w") as fo, open(f"{db}.lookup") as lk:
        for line in fi:
            if line.startswith(">"):
                n_h += 1
                ident = next(lk).split("\t")[1]
                m = re.search(r"(?:OX|TaxID)=(\d+)", line)
                if m:
                    fo.write(f"{ident}\t{m.group(1)}\n")
                else:
                    n_miss += 1
        if next(lk, None) is not None:
            raise RuntimeError("db.lookup has more entries than the FASTA has headers; cannot map taxonomy ids safely.")
    print(f"{n_h:,} sequences; {n_miss:,} headers without OX= or TaxID=")
    if n_h == 0 or n_miss > 0.5 * n_h:
        raise RuntimeError("Most headers carry no taxonomy id (OX= or TaxID=), so taxonomy cannot be built. Use a FASTA with UniProt-style or UniRef-style headers.")
    dump = ensure_taxdump()
    shutil.rmtree(workdir / "tmp_tax", ignore_errors=True)
    mm("createtaxdb", db, workdir / "tmp_tax", "--tax-mapping-file", mapf, "--ncbi-tax-dump", dump, "--threads", THREADS)
    return n_h

def first_sequence(db):
    with open(f"{db}.index") as f:
        key, off, ln = f.readline().split("\t")[:3]
    with open(db, "rb") as f:
        f.seek(int(off))
        return f.read(int(ln)).decode().replace("\x00", "").strip()

def finish_db(outdir, name, kind, source, taxids):
    """Check a finished database folder, write db_info.json and optionally pack it. Returns the info dict."""
    outdir = Path(outdir); db = outdir / "db"
    for ext in (".dbtype", ".index", ".lookup", "_h", "_h.index", "_mapping", "_taxonomy"):
        if not Path(str(db) + ext).exists():
            raise RuntimeError(f"{outdir} is missing db{ext}; the database is incomplete.")
    links = [p.name for p in outdir.iterdir() if p.is_symlink()]
    if links:
        raise RuntimeError(f"{outdir} contains links ({links[:3]}...), so it is not standalone.")
    n = n_seqs(db)
    with open(str(db) + "_mapping") as f:
        n_tax = len({l.split("\t")[1].strip() for l in f if "\t" in l})
    checks = {"standalone": True, "self_test": "not run"}
    if SELF_TEST:
        q = first_sequence(db)
        qf = WORK / "selftest.fa"; qf.write_text(f">selftest\n{q}\n")
        out = WORK / "selftest.tsv"
        shutil.rmtree(TMP / "selftest", ignore_errors=True)
        mm("easy-search", qf, db, out, TMP / "selftest", "-s", 4, "-e", "1e-10", "--max-seqs", 5, "--threads", THREADS, "-v", 1,
           "--format-output", "query,target,fident,taxid,taxlineage")
        rows = [l.rstrip("\n").split("\t") for l in open(out)]
        good = [r for r in rows if len(r) >= 5 and float(r[2]) >= 0.99 and r[3] not in ("", "0") and r[4]]
        if not good:
            raise RuntimeError("Self-test failed: searching a sequence of this database against itself did not return itself with a taxonomy lineage. "
                               f"First rows: {rows[:3]}")
        checks["self_test"] = f"passed (hit {good[0][1]}, taxid {good[0][3]})"
        print("Self-test:", checks["self_test"])
    created = datetime.datetime.now().isoformat(timespec="seconds")
    info = {"format": 1, "name": name, "kind": kind, "source": source, "taxids": taxids, "db_prefix": "db",
            "sequences": n, "distinct_taxids": n_tax, "created": created, "mmseqs_version": MMSEQS_VERSION,
            "notebook": NB_VERSION, "id": keyof(name, n, created), "checks": checks}
    (outdir / "db_info.json").write_text(json.dumps(info, indent=1))
    info["size_gb"] = round(dir_size_gb(outdir), 2)
    print(f"{name}: {n:,} sequences, {n_tax:,} distinct taxids, {info['size_gb']} GB in {outdir}")
    return info

def package(outdir, info):
    outdir = Path(outdir)
    dest = Path(ARCHIVE_DIR) if ARCHIVE_DIR else LIB
    dest.mkdir(parents=True, exist_ok=True)
    tar = dest / f"{outdir.name}.tar"
    if PACKAGE_TAR:
        with tarfile.open(tar, "w") as tf:
            tf.add(outdir, arcname=outdir.name)
        print(f"Packed {tar} ({tar.stat().st_size / 1e9:.2f} GB)")
        if DOWNLOAD_PACKAGE and IN_COLAB:
            from google.colab import files as colab_files
            colab_files.download(str(tar))
    return tar if PACKAGE_TAR else None

print(f"{SYSTEM} | python {platform.python_version()} | MMseqs2 {MMSEQS_VERSION} ({MMSEQS})")
print("threads:", THREADS, "| free disk (GB) in library dir:", round(shutil.disk_usage(LIB).free / 1e9, 1))
print("library:", LIB, "| work:", WORK)

## 2. Query list
**What happens.** Turns your pasted lists into clean, unique IDs: it finds every `IPR` + 6 digits and `PF` + 5 digits (so names glued to an ID, such as `IPR0505372-oxoacid_dehydrogenase`, are split correctly), upper-cases them, removes duplicates and anything in `EXCLUDE_IDS`, and builds the list of taxids. It prints one row per ID with the role it plays in the architecture (L lipoyl, P E3-binding, C catalytic, `family` whole-protein family, `broad` generic superfamily).

**Expect.** Instant. If your list contained the same ID several times, the output says how many duplicates were removed.

**Parameters that matter.** `INTERPRO_IDS`, `PFAM_IDS`, `EXCLUDE_IDS`, and the taxon boxes. An ID that is not in `ENTRY_ROLE` (Advanced cell) is still fetched and listed, but gets role `other` and stays out of the architecture.

In [ ]:
#@title 2. Query list { display-mode: "form" }
import os, sys, re, json, time, ssl, urllib.parse, urllib.request, urllib.error
import pandas as pd

def clean_ids(text):
    return re.findall(r"IPR\d{6}|PF\d{5}|G3DSA:\d+(?:\.\d+)+|SSF\d{5}|PTHR\d{5}(?::SF\d+)?", text.upper())

raw_ipr, raw_pf = clean_ids(INTERPRO_IDS), clean_ids(PFAM_IDS)
excl = set(clean_ids(EXCLUDE_IDS))
SIG = [i for i in dict.fromkeys(clean_ids(SIGNATURE_IDS) + [i for i in raw_ipr + raw_pf if not i.startswith(("IPR", "PF"))]) if i not in excl]
SIGDB = lambda i: "Gene3D" if i.startswith("G3DSA") else "SUPFAM" if i.startswith("SSF") else "PANTHER"
IPR = [i for i in dict.fromkeys(raw_ipr) if i.startswith("IPR") and i not in excl]
PFAM = [i for i in dict.fromkeys(raw_pf) if i.startswith("PF") and i not in excl]
misplaced = [i for i in raw_ipr if i.startswith("PF")] + [i for i in raw_pf if i.startswith("IPR")]
if misplaced: print("NOTE: IDs found in the other list were treated by their prefix:", misplaced)
IPR += [i for i in dict.fromkeys(i for i in raw_pf if i.startswith("IPR")) if i not in IPR and i not in excl]
PFAM += [i for i in dict.fromkeys(i for i in raw_ipr if i.startswith("PF")) if i not in PFAM and i not in excl]
n_dup = len(raw_ipr) + len(raw_pf) - len(set(raw_ipr) | set(raw_pf))
if not IPR and not PFAM and not SIG:
    raise ValueError("No InterPro or Pfam IDs left. Check INTERPRO_IDS, PFAM_IDS and EXCLUDE_IDS.")
PRESET = {"fungi": 4751, "metazoa": 33208, "plants": 33090, "bacteria": 2}
TAXIDS = sorted({PRESET[n] for n, on in [("fungi", TAX_FUNGI), ("metazoa", TAX_METAZOA), ("plants", TAX_PLANTS), ("bacteria", TAX_BACTERIA)] if on}
                | {int(x) for x in re.findall(r"\d+", EXTRA_TAXIDS)})
if not TAXIDS:
    raise ValueError("No taxon selected. Tick a group or give EXTRA_TAXIDS.")
QUERIES = [("InterPro", i, t) for i in IPR for t in TAXIDS] + [("Pfam", i, t) for i in PFAM for t in TAXIDS] + [(SIGDB(i), i, t) for i in SIG for t in TAXIDS]
idtab = pd.DataFrame([{"database": "InterPro" if i.startswith("IPR") else "Pfam" if i.startswith("PF") else SIGDB(i), "id": i, "name": ENTRY_NAME.get(i, ""), "role": ENTRY_ROLE.get(i, "other")}
                      for i in IPR + PFAM + SIG])
display(idtab)
print(f"{len(IPR)} InterPro + {len(PFAM)} Pfam + {len(SIG)} signature IDs ({n_dup} duplicate mentions removed), taxids {TAXIDS}: {len(QUERIES)} queries.")
broad = [i for i in IPR + PFAM + SIG if ENTRY_ROLE.get(i) == "broad"]
if broad: print("Broad entries (may match unrelated proteins; check the counts below):", broad)

## 3. Counts
**What happens.** For every ID × taxon, one small request asks InterPro how many UniProt proteins carry the entry (`page_size=1`, the answer's `count` field). Requests that fail are retried a few times with a pause; the notebook stops with the failing URL if they keep failing. Results are cached in `WORKDIR/cache`.

The table's columns are taxids (4751 = fungi); the row numbers are just row numbers.

**Expect.** A few seconds per query, so under a minute to a few minutes for ~14 IDs × 1 taxon (not measured). Output: a table of counts. Counts are for the whole entry, so the union will be smaller than their sum (proteins carry several of these domains).

**Parameters that matter.** `REVIEWED_ONLY` changes the counts. Tick `COUNT_ONLY` to stop here and check which IDs are huge before downloading. If one ID is much larger than the rest (the `broad` ones are the usual cause), add it to `EXCLUDE_IDS`.

In [ ]:
#@title 3. Counts { display-mode: "form" }
CACHE = WORK / "cache"; CACHE.mkdir(parents=True, exist_ok=True)
CTX = None if VERIFY_SSL else ssl._create_unverified_context()
if not VERIFY_SSL:
    print("WARNING: VERIFY_SSL is off; certificates are not checked.")
SRC = "reviewed" if REVIEWED_ONLY else "UniProt"

def entry_url(db, acc, tax, page_size=PAGE_SIZE, sequence=True):
    path = {"InterPro": f"entry/InterPro/{acc}", "Pfam": f"entry/all/pfam/{acc}", "Gene3D": f"entry/cathgene3d/{acc}",
            "SUPFAM": f"entry/ssf/{acc}", "PANTHER": f"entry/panther/{acc}"}[db]
    q = f"?page_size={page_size}" + ("&extra_fields=sequence" if sequence else "")
    return f"{API_BASE}/protein/{SRC}/{path}/taxonomy/uniprot/{tax}/{q}"

import threading

def _heartbeat(stop, label, every=15):
    t0 = time.time()
    while not stop.wait(every):
        print(f"    ... still waiting for the server ({time.time() - t0:.0f}s) {label}", flush=True)

def http_json(url, retries=4, label="", text=False):
    """GET a JSON document (or, with text=True, a (text, Link header) pair). Returns None for 'no content' (204). Retries timeouts and server errors with a pause.
    While waiting for a slow answer a message is printed every 15 seconds, so a long request does not look like a hang."""
    attempt = 0
    while True:
        stop = threading.Event()
        threading.Thread(target=_heartbeat, args=(stop, label), daemon=True).start()
        try:
            req = urllib.request.Request(url, headers={"Accept": "text/plain" if text else "application/json"})
            with urllib.request.urlopen(req, context=CTX, timeout=300) as r:
                if r.status == 204: return None
                body = r.read().decode()
                return (body, r.headers.get("Link", "")) if text else json.loads(body)
        except urllib.error.HTTPError as e:
            if e.code == 204: return None
            if e.code in (408, 429, 500, 502, 503, 504) and attempt < retries:
                attempt += 1; wait = 20 * attempt; print(f"  HTTP {e.code}, waiting {wait}s before retry {attempt}/{retries}: {url}", flush=True); time.sleep(wait); continue
            raise RuntimeError(f"HTTP {e.code} for {url}")
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            if attempt < retries:
                attempt += 1; wait = 20 * attempt; print(f"  {type(e).__name__}: {e}; waiting {wait}s before retry {attempt}/{retries}: {url}", flush=True); time.sleep(wait); continue
            raise RuntimeError(f"Could not reach {url}: {e}")
        finally:
            stop.set()

def fmt_t(sec):
    sec = int(sec); return f"{sec // 3600}h{sec % 3600 // 60:02d}m" if sec >= 3600 else (f"{sec // 60}m{sec % 60:02d}s" if sec >= 60 else f"{sec}s")

counts = []; T0 = time.time()
for qi, (db, acc, tax) in enumerate(QUERIES, 1):
    cf = CACHE / f"count_{SRC}_{acc}_{tax}.json"
    if cf.exists() and not REFETCH:
        n = json.loads(cf.read_text())["count"]; how = "cached"
    else:
        pl = http_json(entry_url(db, acc, tax, page_size=1, sequence=False), label=f"(count {acc})")
        n = int(pl["count"]) if pl else 0
        cf.write_text(json.dumps({"count": n})); time.sleep(PAUSE_SECONDS); how = "asked server"
    print(f"[{qi}/{len(QUERIES)}] {acc} taxon {tax}: {n:,} proteins ({how}; {fmt_t(time.time() - T0)} elapsed)", flush=True)
    counts.append({"database": db, "id": acc, "name": ENTRY_NAME.get(acc, ""), "role": ENTRY_ROLE.get(acc, "other"), "taxid": tax, "proteins": n})
CT = pd.DataFrame(counts)
_pv = CT.pivot_table(index=["database", "id", "name", "role"], columns="taxid", values="proteins", aggfunc="sum")
_pv.columns = [f"proteins, taxid {c}" for c in _pv.columns]
display(_pv.reset_index())
print(f"Sum over queries: {CT.proteins.sum():,} protein hits (the union is smaller). Cache: {CACHE}")
if COUNT_ONLY:
    print("COUNT_ONLY is ticked: stopping here. Untick it to fetch.")

## 4. Fetch and union
**What happens.** For each ID × taxon (except `broad` entries, unless `FETCH_BROAD` is ticked) the notebook downloads all pages (`page_size` proteins each, following the `next` link) with the domain positions (and, if `SEQUENCES_FROM_UNIPROT` is off, the sequence), caches the answer in `WORKDIR/cache`, and merges everything by UniProt accession. Each protein ends up with: name, organism taxid, length, the list of query IDs that found it, and all domain positions found (entry, start, end). If a protein's taxid is missing in the API answer, it is looked up in UniProt in batches.

**Sequences.** By default (`SEQUENCES_FROM_UNIPROT`) the InterPro pages carry no sequences, which makes them much faster. When all queries are done, the sequences of the union are downloaded from UniProt, 200 accessions per request, with a progress line per batch and a cache (`uniprot_sequences.tsv`) so an interrupted run continues. Accessions UniProt no longer has (obsolete or merged) are listed in `no_sequence.txt` and dropped. The InterPro server caps pages at 200 proteins whatever `PAGE_SIZE` says.

**Progress.** After every page the cell prints the query number, the page and protein counts, the elapsed time and a rough time left for that query, and after every query the size of the union so far. While waiting for a slow answer it prints a "still waiting" line every 15 seconds, so silence means something is wrong. Queries read from the cache are listed as such. The count step prints one line per query in the same way.

**Expect.** About one request per 200 proteins plus `PAUSE_SECONDS` between requests, so a few thousand proteins take a few minutes (not measured). Re-running reads the cache and takes seconds, unless `REFETCH` is ticked. A failed query stops the run with its URL; fix or exclude it and run again, the finished queries stay cached.

**Parameters that matter.** `FETCH_BROAD` (off by default; one broad entry such as the CAT-like superfamily can be several times larger than all the others together); the taxon boxes and ID lists decide what is fetched; `REVIEWED_ONLY` keeps only Swiss-Prot entries (small, curated, much less complete); `MIN_LENGTH` removes short fragments; `PAUSE_SECONDS` is the politeness delay.

In [ ]:
#@title 4. Fetch and union { display-mode: "form" }
def fill_taxids(P, accs):
    """Fill missing taxids of the proteins `accs` in dict P from UniProt (batches of 100). Returns the accessions still without taxid."""
    accs = list(accs)
    for i in range(0, len(accs), 100):
        batch = accs[i:i + 100]
        q = "(" + " OR ".join(f"accession:{a}" for a in batch) + ")"
        url = f"{UNIPROT_BASE}/uniprotkb/search?query={urllib.parse.quote(q)}&format=tsv&fields=accession,organism_id&size=500"
        with urllib.request.urlopen(urllib.request.Request(url), context=CTX, timeout=120) as r:
            for line in r.read().decode().splitlines()[1:]:
                a, _, t = line.partition("\t")
                if a in P and t.strip(): P[a]["taxid"] = t.strip()
        print(f"  taxid lookup: {min(i + 100, len(accs)):,}/{len(accs):,}", flush=True)
        time.sleep(PAUSE_SECONDS)
    return [a for a in accs if not P[a]["taxid"]]

if COUNT_ONLY:
    print("COUNT_ONLY is ticked: skipped.")
else:
    def fetch_query(db, acc, tax, tag=""):
        cf_full = CACHE / f"proteins_{SRC}_{acc}_{tax}.jsonl"                     # with sequences
        cf_noseq = CACHE / f"proteins_{SRC}_{acc}_{tax}_noseq.jsonl"              # without sequences
        cf = cf_noseq if SEQUENCES_FROM_UNIPROT else cf_full
        for c in ([cf_full, cf_noseq] if SEQUENCES_FROM_UNIPROT else [cf_full]):  # a cache with sequences is always usable
            if c.exists() and not REFETCH:
                rows = [json.loads(l) for l in c.read_text().splitlines() if l.strip()]
                if tag: print(f"{tag} {acc} taxon {tax}: {len(rows):,} proteins from cache", flush=True)
                return rows
        out, url, page, total, t0 = [], entry_url(db, acc, tax, sequence=not SEQUENCES_FROM_UNIPROT), 0, None, time.time()
        while url:
            pl = http_json(url, label=f"({acc}, page {page + 1})")
            if not pl: break
            page += 1
            if total is None: total = pl.get("count")
            if page == 1:
                got = len(pl.get("results", []))
                eff = got if (pl.get("next") and got) else PAGE_SIZE
                if pl.get("next") and got < PAGE_SIZE:
                    print(f"{tag} NOTE: asked for {PAGE_SIZE} proteins per page, the server returns {got}; it caps the page size, so a larger PAGE_SIZE does not help. Estimates use {got}.", flush=True)
            for it in pl.get("results", []):
                md = it.get("metadata", {})
                org = md.get("source_organism") or {}
                doms = []
                for e in it.get("entries") or []:
                    for loc in e.get("entry_protein_locations") or []:
                        for fr in loc.get("fragments") or []:
                            doms.append([e.get("accession", acc), int(fr["start"]), int(fr["end"])])
                out.append({"acc": md.get("accession"), "name": md.get("name") or "", "taxid": str(org.get("taxId") or ""),
                            "length": md.get("length"), "seq": (it.get("extra_fields") or {}).get("sequence") or "", "found_by": acc, "domains": doms})
            url = pl.get("next")
            el = time.time() - t0
            if total:
                pages_total = -(-int(total) // max(eff, 1))
                eta = el / page * (pages_total - page)
                print(f"{tag} {acc} taxon {tax}: page {page}/{pages_total}, {len(out):,}/{int(total):,} proteins, {fmt_t(el)} elapsed, about {fmt_t(eta)} left for this query", flush=True)
            else:
                print(f"{tag} {acc} taxon {tax}: page {page}, {len(out):,} proteins, {fmt_t(el)} elapsed", flush=True)
            if url: time.sleep(PAUSE_SECONDS)
        cf.write_text("\n".join(json.dumps(o) for o in out) + ("\n" if out else ""))
        print(f"{tag} {acc} taxon {tax}: done, {len(out):,} proteins in {page} page(s), {fmt_t(time.time() - t0)}", flush=True)
        return out

    FETCH_Q = [q for q in QUERIES if FETCH_BROAD or ENTRY_ROLE.get(q[1]) != "broad"]
    skipped = sorted({q[1] for q in QUERIES} - {q[1] for q in FETCH_Q})
    if skipped: print("Counted but not fetched (role 'broad', FETCH_BROAD is off):", skipped)
    P = {}; T1 = time.time()
    print(f"Fetching {len(FETCH_Q)} queries. Each page holds up to {PAGE_SIZE} proteins" + ("" if SEQUENCES_FROM_UNIPROT else " with sequences") + "; a page can take several seconds. "
          "Progress is printed after every page; if nothing appears for a minute, a 'still waiting' line is printed every 15 s.", flush=True)
    for qi, (db, acc, tax) in enumerate(FETCH_Q, 1):
        for r in fetch_query(db, acc, tax, tag=f"[{qi}/{len(FETCH_Q)}]"):
            if not r["acc"]: continue
            p = P.setdefault(r["acc"], {"acc": r["acc"], "name": r["name"], "taxid": r["taxid"], "length": r["length"], "seq": r["seq"], "found_by": set(), "domains": set()})
            p["found_by"].add(r["found_by"])
            for d in r["domains"]: p["domains"].add(tuple(d))
            if not p["seq"] and r["seq"]: p["seq"] = r["seq"]
            if not p["taxid"] and r["taxid"]: p["taxid"] = r["taxid"]
        print(f"      union so far: {len(P):,} distinct proteins ({fmt_t(time.time() - T1)} since the fetch started)", flush=True)
    print(f"Union: {len(P):,} distinct proteins from {len(FETCH_Q)} queries (sum of query sizes: {sum(len(fetch_query(*q)) for q in FETCH_Q):,}).")
    def fetch_sequences(P, accs, batch=200):
        """Download sequence and taxid of the proteins `accs` from UniProt, `batch` accessions per request. Results are cached
        in WORKDIR/cache/uniprot_sequences.tsv, so an interrupted run continues where it stopped."""
        sc = CACHE / "uniprot_sequences.tsv"
        known = {}
        if sc.exists() and not REFETCH:
            for l in sc.read_text().splitlines():
                parts = l.split("\t")
                if len(parts) == 3: known[parts[0]] = (parts[1], parts[2])
        todo = [a for a in accs if a not in known]
        print(f"Sequences: {len(accs) - len(todo):,} from cache, {len(todo):,} to download from UniProt in {-(-len(todo) // batch)} batches of {batch}.", flush=True)
        t0 = time.time(); n = 0
        for i in range(0, len(todo), batch):
            part = todo[i:i + batch]
            url = f"{UNIPROT_BASE}/uniprotkb/accessions?accessions={','.join(part)}&format=tsv&fields=accession,organism_id,sequence&size=500"
            got = {}
            while url:
                res = http_json(url, label=f"(UniProt sequences, batch {i // batch + 1})", text=True)
                if not res: break
                body, link = res
                for line in body.splitlines()[1:]:
                    c = line.split("\t")
                    if len(c) >= 3 and c[2].strip(): got[c[0]] = (c[1].strip(), c[2].strip())
                m = re.search(r'<([^>]+)>;\s*rel="next"', link or "")
                url = m.group(1) if m else None
            with open(sc, "a") as fh:
                for a, (tx, sq) in got.items(): fh.write(f"{a}\t{tx}\t{sq}\n")
            known.update(got); n += len(part)
            el = time.time() - t0
            print(f"  sequences: {n:,}/{len(todo):,} requested, {len(got)} of this batch returned, {fmt_t(el)} elapsed, about {fmt_t(el / n * (len(todo) - n))} left", flush=True)
            time.sleep(PAUSE_SECONDS)
        for a in accs:
            if a in known:
                P[a]["seq"] = P[a]["seq"] or known[a][1]
                if not P[a]["taxid"]: P[a]["taxid"] = known[a][0]
        return [a for a in accs if not P[a]["seq"]]

    no_seq = [a for a, p in P.items() if not p["seq"]]
    if no_seq and SEQUENCES_FROM_UNIPROT:
        no_seq = fetch_sequences(P, no_seq)
        if no_seq:
            print(f"WARNING: UniProt returned no sequence for {len(no_seq):,} accessions (obsolete or merged entries, for example {no_seq[:3]}); they are dropped from the union.")
            OUT.mkdir(parents=True, exist_ok=True); (OUT / "no_sequence.txt").write_text("\n".join(no_seq) + "\n")
            for a in no_seq: P.pop(a)
            no_seq = []
    if no_seq:
        raise RuntimeError(f"{len(no_seq)} proteins came without a sequence (for example {no_seq[:3]}). The API answer did not contain extra_fields.sequence; "
                           "check the cached files in " + str(CACHE) + " and the URL pattern in entry_url().")
    # fill missing taxids from UniProt
    miss = [a for a, p in P.items() if not p["taxid"]]
    if miss:
        print(f"{len(miss):,} proteins have no taxid in the API answer; asking UniProt.")
        still = fill_taxids(P, miss)
        if still: print(f"WARNING: {len(still)} proteins still have no taxid; they are kept in the tables but left out of the database.")
    if MIN_LENGTH:
        before = len(P); P = {a: p for a, p in P.items() if len(p["seq"]) >= MIN_LENGTH}
        print(f"MIN_LENGTH {MIN_LENGTH}: removed {before - len(P):,} shorter proteins.")

## 5. Architecture
**What happens.** For each protein, the domain positions are grouped by role. Overlapping regions of the same role (for example a Pfam and an InterPro lipoyl match on the same stretch) are merged into one region. The regions are ordered by position and written as a string, such as `L-P-C` (lipoyl, E3-binding, catalytic) or `L2-P-C` (two lipoyl domains). A **preliminary class** follows from which of L, P and C are present:
- `L-P-C` present: **E2-like** (typical PDC E2)
- L and P, no C: **E3BP / protein X-like**
- L and C, no P: **2-oxoacid E2 without PSBD** (OGDC- or BCKDC-type, but also diverged E2s)
- P only, L only, C only, or P with C: as named
- none of the three located: **no core domain**

**Expect.** Seconds. Output: a table of classes and the most common architectures, with protein and species counts, plus a bar chart.

**Names are not reliable.** Some submitted names are wrong: a fungal protein X (E3BP) can be called "pyridoxine biosynthesis protein" because both use the gene symbol PDX1. Classify by domains and sequence, not by name. The table flags such cases in a `name_note` column.

**Files for inspection.** With `WRITE_ARCH_FILES`, `OUT_DIR/architectures/` gets one FASTA per class and architecture (numbered by size, e.g. `01_E2_like_L_P_C__L-P-C.fasta`), `rare.fasta` for architectures with fewer than `ARCH_MIN_SIZE` proteins, and `architectures.tsv` (one row per architecture: proteins, taxids, length range, three example accessions, file name). FASTA headers carry the class, architecture, length and all domain positions, so a file can be opened, aligned or searched on its own.

**Parameters that matter.** `ENTRY_ROLE` (Advanced cell) decides which entry counts as L, P or C. These classes are labels from annotation, not functional assignments: a diverged E3BP that lost its detectable E3-binding match is labelled "L only", and some E2s lack a located PSBD. Use the table to see what exists and notebook C to separate E2 from E3BP.

In [ ]:
#@title 5. Architecture { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

def merge_role(intervals):
    out = []
    for s, e in sorted(intervals):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out

def architecture(doms):
    regions = []
    for role in ("L", "P", "C"):
        iv = [(s, e) for acc, s, e in doms if ENTRY_ROLE.get(acc) == role]
        regions += [(s, e, role) for s, e in merge_role(iv)]
    regions.sort()
    toks, i = [], 0
    while i < len(regions):
        j = i
        while j + 1 < len(regions) and regions[j + 1][2] == regions[i][2]: j += 1
        toks.append(regions[i][2] + (str(j - i + 1) if j > i else "")); i = j + 1
    return "-".join(toks), regions

def classify(roles):
    L, Pp, C = "L" in roles, "P" in roles, "C" in roles
    if L and Pp and C: return "E2-like (L-P-C)"
    if L and Pp: return "E3BP / protein X-like (L-P)"
    if L and C: return "2-oxoacid E2 without PSBD (L-C)"
    if Pp and C: return "P-C"
    if Pp: return "PSBD only"
    if L: return "lipoyl only"
    if C: return "catalytic only"
    return "no core domain located"

def build_pt(P):
    rows = []
    for a, p in P.items():
        arch, regs = architecture(p["domains"])
        roles = {r[2] for r in regs}
        cl = classify(roles) if p["domains"] or not p.get("added") else "added from reference (no domain annotation)"
        rows.append({"acc": a, "taxid": p["taxid"], "name": p["name"], "length": len(p["seq"]), "architecture": arch, "class": cl,
                     "found_by": ",".join(sorted(p["found_by"])), "n_found_by": len(p["found_by"]),
                     "domains": ";".join(f"{acc}:{s}-{e}" for acc, s, e in sorted(p["domains"], key=lambda d: (d[1], d[2], d[0])))})
    PT = pd.DataFrame(rows)
    sus = PT.name.str.contains(r"pyridox|vitamin b6|pyridoxal", case=False, regex=True) & PT["class"].str.contains("E2-like|E3BP|PSBD|lipoyl", regex=True)
    PT["name_note"] = np.where(sus, "name says pyridoxine/B6 but domains are PDC-type (PDX1 gene-symbol clash?)", "")
    return PT, int(sus.sum())

def write_architecture_files(P, PT, verbose=True):
    """One FASTA per (class, architecture) with at least ARCH_MIN_SIZE proteins, the rest pooled in rare.fasta, and architectures.tsv
    (all architectures). Headers: >ACC name OX=taxid class=... arch=... len=... domains=ENTRY:start-end;..."""
    import shutil
    d = OUT / "architectures"
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
    def wrap(s, n=80): return "\n".join(s[i:i + n] for i in range(0, len(s), n))
    def slug(x): return re.sub(r"[^A-Za-z0-9]+", "_", x).strip("_")[:40] or "none"
    g = (PT.groupby(["class", "architecture"]).agg(proteins=("acc", "size"), taxids=("taxid", "nunique"), median_length=("length", "median"),
                                                    min_length=("length", "min"), max_length=("length", "max"), examples=("acc", lambda s: ",".join(list(s)[:3])))
         .reset_index().sort_values("proteins", ascending=False).reset_index(drop=True))
    files, rare = [], []
    for i, r in g.iterrows():
        files.append(f"{i + 1:02d}_{slug(r['class'])}__{slug(r['architecture'])}.fasta" if r["proteins"] >= ARCH_MIN_SIZE else "rare.fasta")
    g["file"] = files
    fmap = {(r["class"], r["architecture"]): r["file"] for _, r in g.iterrows()}
    handles = {}
    try:
        for _, row in PT.iterrows():
            fn = fmap[(row["class"], row["architecture"])]
            if fn not in handles: handles[fn] = open(d / fn, "w", newline="\n")
            p = P[row["acc"]]
            nm = re.sub(r"[\r\n]+", " ", str(row["name"]))
            handles[fn].write(f">{row['acc']} {nm} OX={row['taxid']} class={row['class'].replace(' ', '_')} arch={row['architecture'] or 'none'} len={row['length']} domains={row['domains']}\n{wrap(p['seq'])}\n")
    finally:
        for h in handles.values(): h.close()
    g.to_csv(d / "architectures.tsv", sep="\t", index=False)
    if verbose:
        big = g[g.file != "rare.fasta"]
        print(f"Architecture files in {d}: {len(big)} FASTA files for architectures with >= {ARCH_MIN_SIZE} proteins "
              f"({int(big.proteins.sum()):,} proteins), rare.fasta for {int((g.file == 'rare.fasta').sum())} smaller architectures "
              f"({int(g[g.file == 'rare.fasta'].proteins.sum()):,} proteins), and architectures.tsv with all {len(g)}.")
    return g

if COUNT_ONLY:
    print("COUNT_ONLY is ticked: skipped.")
else:
    PT, n_sus = build_pt(P)
    if n_sus: print(f"{n_sus:,} proteins are named as pyridoxine/vitamin B6 proteins but carry lipoyl/PSBD domains; see the name_note column. Submitter names are unreliable here.")
    cls = PT.groupby("class").agg(proteins=("acc", "size"), species_taxids=("taxid", "nunique"), median_length=("length", "median")).sort_values("proteins", ascending=False)
    display(cls)
    arch = PT.groupby(["class", "architecture"]).agg(proteins=("acc", "size"), taxids=("taxid", "nunique")).sort_values("proteins", ascending=False).reset_index()
    print("Most common architectures:"); display(arch.head(25))
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(cls) + 1.2))
    ax.barh(cls.index[::-1], cls.proteins[::-1], color="#3b6ea8")
    for y, (v, s) in enumerate(zip(cls.proteins[::-1], cls.species_taxids[::-1])): ax.text(v, y, f"  {v:,} ({s:,} taxids)", va="center", fontsize=8)
    ax.set_xlabel("proteins"); ax.set_title("Preliminary architecture classes"); ax.margins(x=0.25)
    plt.tight_layout(); plt.show()
    if WRITE_ARCH_FILES:
        ARCH = write_architecture_files(P, PT)
        display(ARCH.drop(columns=["examples"]).head(30))
    broad_only = PT[(PT["class"] == "no core domain located")]
    print(f"{len(broad_only):,} proteins have no L, P or C domain located (found only through family or broad entries). Look at them before using the full set.")

## 6. Check against a reference set (optional)
**What happens.** Compares the union built so far with a list of sequences you supply (for example the ~35 UniRef50 hits, your PSI-BLAST hits, or sequences you know are PX/E3BP). Each reference sequence is matched to the union in up to three ways, in this order:
1. **Accession.** The UniProt accession is read from the header (`sp|ACC|...`, `tr|ACC|...`, `UniRef50_ACC`, or a bare accession as the first word) and looked up in the union. UniParc IDs (`UPI...`) are recognised too, but the union only holds UniProtKB entries, so they match only by sequence.
2. **Identical sequence.** Same amino-acid sequence (case and a trailing `*` ignored), whatever the name. This catches RefSeq or GenBank copies of UniProt entries.
3. **Similar sequence** (`MATCH_SIMILAR`, needs MMseqs2). Sequences not yet matched are searched against the union; a hit with identity ≥ `MATCH_IDENTITY` and coverage ≥ `MATCH_COVERAGE` of both sequences counts as the same protein (an isoform, strain variant or fragment).

The result is a **Venn diagram** (two circles: the union and your reference set) with the counts, and two tables: matched and **missing** reference sequences. Circle areas are only roughly proportional (the smaller circle is not drawn below a minimum size). With `DIAGNOSE_MISSING`, the notebook asks UniProt for the InterPro and Pfam entries of up to 50 missing reference proteins (those with a UniProt accession) and tells you why each was missed: no query entry matched it, or it has a query entry but was left out (other taxon, reviewed-only filter, entry not fetched).

With `INCLUDE_MISSING`, the missing reference sequences are **added to the union**, so they appear in the output files and in the database. They are labelled `reference:<file>` in `found_by` and have no domain annotation. Their taxid is taken from `OX=`/`TaxID=` in the header, or from UniProt for accessions; a sequence without a taxid is kept in the tables but left out of the database.

**Expect.** Seconds for the comparison. The similarity search takes seconds to a minute for a few hundred sequences. Diagnosis makes one UniProt request per missing protein (up to 50), about a second each (not measured).

**Parameters that matter.** `REFERENCE_FASTA` (path), or `UPLOAD_REFERENCE` in Colab, or `REFERENCE_INLINE` in the Advanced cell; `MATCH_SIMILAR`, `MATCH_IDENTITY`, `MATCH_COVERAGE` (looser values match more reference sequences to the union); `INCLUDE_MISSING`; `DIAGNOSE_MISSING`. Leave `REFERENCE_FASTA` empty to skip this step. Many reference proteins missing from the union means the annotation misses them; this is the check for what annotation-based collection cannot reach.

In [ ]:
#@title 6. Check against a reference set { display-mode: "form" }
UNIPROT_ACC = re.compile(r"^(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})$")

def ref_accession(header):
    first = header.split()[0] if header.split() else ""
    mu = re.match(r"^(?:UniParc\|)?(UPI[0-9A-F]{10})$", first.split("|")[0] if first.startswith("UPI") else first)
    if mu: return mu.group(1)
    m = re.match(r"^(?:sp|tr)\|([A-Z0-9]+)\|", first) or re.match(r"^UniRef\d+_([A-Z0-9]+)$", first)
    cand = m.group(1) if m else first.split("|")[0].split(".")[0]
    return cand if UNIPROT_ACC.match(cand) else ""

def read_reference():
    if UPLOAD_REFERENCE:
        if not IN_COLAB: raise RuntimeError("UPLOAD_REFERENCE only works in Colab; give a path in REFERENCE_FASTA.")
        from google.colab import files as colab_files
        up = colab_files.upload(); return list(up.values())[0].decode(), "uploaded"
    if REFERENCE_FASTA:
        p = Path(REFERENCE_FASTA)
        if not p.exists(): raise FileNotFoundError(f"REFERENCE_FASTA {p} does not exist.")
        return p.read_text(), p.name
    if REFERENCE_INLINE.strip(): return REFERENCE_INLINE, "inline"
    return None, None

ref_text, ref_label = (None, None) if COUNT_ONLY else read_reference()
if ref_text is None:
    print("No reference set given (REFERENCE_FASTA is empty): skipped.")
else:
    REF = []
    hdr, buf = None, []
    for line in ref_text.splitlines() + [">"]:
        if line.startswith(">"):
            if hdr is not None and buf: REF.append({"header": hdr, "seq": re.sub(r"[^A-Za-z]", "", "".join(buf)).upper()})
            hdr, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    REF = [r for r in REF if r["seq"]]
    if not REF: raise ValueError("The reference FASTA has no sequences.")
    for k, r in enumerate(REF):
        r["id"] = (r["header"].split() or [f"ref{k + 1}"])[0]
        r["acc"] = ref_accession(r["header"])
    print(f"Reference set: {len(REF):,} sequences from {ref_label}; {sum(1 for r in REF if r['acc']):,} have a UniProt accession in the header.")
    by_hash = {}
    for a, p in P.items(): by_hash.setdefault(hash(p["seq"].upper().rstrip("*")), []).append(a)
    for r in REF:
        r["match"], r["matched_to"], r["detail"] = "", "", ""
        if r["acc"] and r["acc"] in P:
            r["match"], r["matched_to"] = "accession", r["acc"]
        else:
            for a in by_hash.get(hash(r["seq"]), []):
                if P[a]["seq"].upper().rstrip("*") == r["seq"]:
                    r["match"], r["matched_to"] = "identical sequence", a; break
    todo = [r for r in REF if not r["match"]]
    if todo and MATCH_SIMILAR:
        mm_exe = MMSEQS or find_mmseqs()
        if not mm_exe:
            print("MATCH_SIMILAR is ticked but MMseqs2 was not found; skipping the similarity step.")
        else:
            MMSEQS = mm_exe
            sim = WORK / "refcheck"; sim.mkdir(exist_ok=True)
            (sim / "union.fa").write_text("".join(f">{a}\n{p['seq']}\n" for a, p in P.items()))
            (sim / "todo.fa").write_text("".join(f">r{i}\n{r['seq']}\n" for i, r in enumerate(todo)))
            shutil.rmtree(sim / "tmp", ignore_errors=True)
            mm("easy-search", sim / "todo.fa", sim / "union.fa", sim / "hits.tsv", sim / "tmp", "--min-seq-id", MATCH_IDENTITY, "-c", MATCH_COVERAGE,
               "--cov-mode", 0, "-e", 1e-5, "--threads", THREADS, "-v", 1, "--format-output", "query,target,fident", quiet=True)
            best = {}
            if (sim / "hits.tsv").exists():
                for line in open(sim / "hits.tsv"):
                    q, t, f = line.rstrip("\n").split("\t")
                    if q not in best or float(f) > best[q][1]: best[q] = (t, float(f))
            for i, r in enumerate(todo):
                if f"r{i}" in best:
                    t, f = best[f"r{i}"]; r["match"], r["matched_to"], r["detail"] = "similar sequence", t, f"{f:.0%} identity"
    matched = [r for r in REF if r["match"]]; missing = [r for r in REF if not r["match"]]
    union_matched = len({r["matched_to"] for r in matched})
    n_union, n_ref, n_both = len(P), len(REF), len(matched)
    print(f"Reference sequences found in the union: {n_both:,}/{n_ref:,} "
          f"(by accession {sum(r['match'] == 'accession' for r in REF):,}, identical sequence {sum(r['match'] == 'identical sequence' for r in REF):,}, "
          f"similar {sum(r['match'] == 'similar sequence' for r in REF):,}); missing: {len(missing):,}.")
    print(f"Union entries matched by the reference: {union_matched:,} of {n_union:,}.")

    # Venn diagram (two circles, areas roughly proportional)
    import math
    from matplotlib.patches import Circle
    def lens(r1, r2, d):
        if d >= r1 + r2: return 0.0
        if d <= abs(r1 - r2): return math.pi * min(r1, r2) ** 2
        a = r1 ** 2 * math.acos((d * d + r1 ** 2 - r2 ** 2) / (2 * d * r1)); b = r2 ** 2 * math.acos((d * d + r2 ** 2 - r1 ** 2) / (2 * d * r2))
        return a + b - 0.5 * math.sqrt((-d + r1 + r2) * (d + r1 - r2) * (d - r1 + r2) * (d + r1 + r2))
    ra = 1.0; rb = max(0.22, math.sqrt(n_ref / max(n_union, 1)))
    target = math.pi * ra ** 2 * (min(n_both, n_union) / max(n_union, 1))
    lo, hi = abs(ra - rb), ra + rb
    for _ in range(60):
        mid = (lo + hi) / 2
        if lens(ra, rb, mid) > target: lo = mid
        else: hi = mid
    d = hi if n_both else ra + rb + 0.15
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.add_patch(Circle((0, 0), ra, color="#3b6ea8", alpha=0.45)); ax.add_patch(Circle((d, 0), rb, color="#d98a2b", alpha=0.5))
    ax.text(-ra * 0.55, 0, f"union only\n{n_union - union_matched:,}", ha="center", va="center", fontsize=11)
    ax.text(d + rb * 0.45 + (0.25 if rb < 0.4 else 0), rb * 1.1 + 0.05, f"reference only\n{len(missing):,}", ha="center", va="bottom", fontsize=11)
    ax.text((d - rb + ra) / 2 if n_both else d, 0, f"both\n{n_both:,}", ha="center", va="center", fontsize=10 if n_both else 9)
    ax.set_title(f"Union so far ({n_union:,} proteins) versus reference set {ref_label} ({n_ref:,} sequences)", fontsize=10)
    ax.set_xlim(-1.15, d + max(rb, 0.35) + 0.9); ax.set_ylim(-1.2, 1.4); ax.set_aspect("equal"); ax.axis("off")
    ax.text(0, -ra - 0.12, "union", ha="center", va="top", color="#3b6ea8"); ax.text(d, -rb - 0.12, "reference", ha="center", va="top", color="#d98a2b")
    ax.text(0.5, 0.0, "", transform=ax.transAxes)
    plt.tight_layout(); plt.show()
    print("'both' counts reference sequences; if several reference sequences match one union entry, the union-side count is smaller (above).")

    MISS = pd.DataFrame([{"id": r["id"], "accession": r["acc"], "length": len(r["seq"]), "header": r["header"]} for r in missing])
    MATCHED = pd.DataFrame([{"id": r["id"], "match": r["match"], "matched_to": r["matched_to"], "detail": r["detail"]} for r in matched])
    if len(MISS):
        # why were they missed? (UniProt cross-references of up to 50 missing proteins that have an accession)
        if DIAGNOSE_MISSING:
            fetched_ids = set(IPR + PFAM + SIG) - ({i for i in IPR + PFAM + SIG if ENTRY_ROLE.get(i) == "broad"} if not FETCH_BROAD else set())
            why, k = {}, 0
            for r in missing:
                if not r["acc"] or k >= 50: continue
                k += 1
                if r["acc"].startswith("UPI"):
                    why[r["id"]] = "UniParc ID: the InterPro fetch covers UniProtKB only, so this is missed unless a UniProtKB entry exists (check the UniParc page for a UniProtKB cross-reference)"
                    continue
                try:
                    with urllib.request.urlopen(urllib.request.Request(f"{UNIPROT_BASE}/uniprotkb/{r['acc']}.json"), context=CTX, timeout=60) as resp:
                        js = json.loads(resp.read().decode())
                    xr = {c["id"] for c in js.get("uniProtKBCrossReferences", []) if c.get("database") in ("InterPro", "Pfam", "Gene3D", "SUPFAM", "PANTHER")}
                    tx = str((js.get("organism") or {}).get("taxonId", ""))
                    hit_q, hit_other = sorted(xr & set(IPR + PFAM + SIG)), sorted(xr - set(IPR + PFAM + SIG))
                    if not xr: why[r["id"]] = f"no InterPro/Pfam entries in UniProt (taxid {tx})"
                    elif not hit_q: why[r["id"]] = f"none of the query entries; has {', '.join(hit_other[:6])} (taxid {tx})"
                    elif not (set(hit_q) & fetched_ids): why[r["id"]] = f"only broad entries not fetched: {', '.join(hit_q)} (taxid {tx})"
                    else: why[r["id"]] = f"has query entries {', '.join(hit_q)} but is not in the union: other taxon, reviewed-only filter, or fetch gap (taxid {tx})"
                except Exception as e:
                    why[r["id"]] = f"UniProt lookup failed ({type(e).__name__})"
                time.sleep(PAUSE_SECONDS)
                if k % 10 == 0: print(f"  diagnosed {k} missing proteins", flush=True)
            MISS["why_missed"] = MISS["id"].map(why).fillna("")
        print("Missing reference sequences:"); display(MISS)
        MISS.to_csv(OUT / "reference_missing.tsv", sep="\t", index=False)
    MATCHED.to_csv(OUT / "reference_matched.tsv", sep="\t", index=False)

    if INCLUDE_MISSING and missing:
        added, no_tax = [], []
        for r in missing:
            m = re.search(r"(?:OX|TaxID)=(\d+)", r["header"])
            key = r["acc"] or re.sub(r"[^\w.\-]", "_", r["id"])
            if key in P: key = key + "_ref"
            nm = " ".join(r["header"].split()[1:]) or r["id"]
            P[key] = {"acc": key, "name": nm, "taxid": m.group(1) if m else "", "length": len(r["seq"]), "seq": r["seq"],
                      "found_by": {"reference:" + str(ref_label)}, "domains": set(), "added": True}
            added.append(key)
        need = [k for k in added if not P[k]["taxid"] and UNIPROT_ACC.match(k)]
        if need:
            print(f"Looking up taxids of {len(need)} added proteins in UniProt.")
            fill_taxids(P, need)
        no_tax = [k for k in added if not P[k]["taxid"]]
        PT, _ = build_pt(P)
        print(f"INCLUDE_MISSING: added {len(added):,} reference sequences to the union (now {len(P):,}). {len(no_tax)} have no taxid and will be left out of the database.")
    elif INCLUDE_MISSING:
        print("INCLUDE_MISSING is ticked but nothing is missing.")
    else:
        print("Missing sequences were not added to the union (INCLUDE_MISSING is off).")

## 7. Output files
**What happens.** Writes to `OUT_DIR` (default `work_family/family_fetch_results`):
- `proteins.fasta`: one entry per protein, header `>ACC name OX=taxid` (the format the database builder and notebooks B and C read).
- `architectures/`: one FASTA per class and architecture plus `architectures.tsv` (see step 5; rewritten here so it matches the final union).
- `proteins.tsv`: accession, taxid, name, length, architecture, class, which IDs found it, a name warning if any.
- `domains.tsv`: one row per domain match (accession, entry, role, start, end).
- `regions_L.fasta`, `regions_P.fasta`, `regions_C.fasta`: the sequence of each lipoyl, E3-binding and catalytic region, named `ACC/start-end`. These are the inputs for domain-wise alignments (see step 8).
- `query_log.tsv`: the count table and the settings used.

**Expect.** Seconds.

**Parameters that matter.** `OUT_DIR`.

In [ ]:
#@title 7. Output files { display-mode: "form" }
if COUNT_ONLY:
    print("COUNT_ONLY is ticked: skipped.")
else:
    def wrap(s, n=80): return "\n".join(s[i:i + n] for i in range(0, len(s), n))
    with open(OUT / "proteins.fasta", "w", newline="\n") as f:
        for a, p in P.items():
            if p["taxid"]:
                f.write(f">{a} {re.sub(r'[\r\n]+', ' ', p['name'])} OX={p['taxid']}\n{wrap(p['seq'])}\n")
    PT.to_csv(OUT / "proteins.tsv", sep="\t", index=False)
    if WRITE_ARCH_FILES: write_architecture_files(P, PT, verbose=False)   # refresh: INCLUDE_MISSING may have changed the union
    dr = [{"acc": a, "entry": acc, "role": ENTRY_ROLE.get(acc, "other"), "start": s, "end": e} for a, p in P.items() for acc, s, e in sorted(p["domains"])]
    pd.DataFrame(dr).to_csv(OUT / "domains.tsv", sep="\t", index=False)
    nreg = {}
    for role in ("L", "P", "C"):
        n = 0
        with open(OUT / f"regions_{role}.fasta", "w", newline="\n") as f:
            for a, p in P.items():
                _, regs = architecture(p["domains"])
                for s, e, r in regs:
                    if r == role:
                        f.write(f">{a}/{s}-{e} {role}\n{wrap(p['seq'][s - 1:e])}\n"); n += 1
        nreg[role] = n
    CT.assign(reviewed_only=REVIEWED_ONLY, excluded=",".join(sorted(excl))).to_csv(OUT / "query_log.tsv", sep="\t", index=False)
    print("Written to", OUT)
    print(f"  proteins.fasta: {sum(1 for p in P.values() if p['taxid']):,} sequences | regions: lipoyl {nreg['L']:,}, E3-binding {nreg['P']:,}, catalytic {nreg['C']:,}")

## 8. Build a database (optional)
**What happens.** Turns `proteins.fasta` into a standalone MMseqs2 database with taxonomy, in the same format as notebook A: `createdb`, a taxid mapping read from the `OX=` field, `createtaxdb` (the NCBI taxonomy is downloaded once, unless `NCBI_TAXDUMP` is given), checks (all files present, no links, counts), a self-test search, and optionally a `.tar`. The folder is `LIBRARY_DIR/<DB_NAME>/`. In notebooks B and C set `DATABASE` to this folder or the `.tar`.

**Expect.** For a few thousand sequences: seconds for the conversion and the self-test, plus the taxonomy download (about 60 MB; not measured here). Needs MMseqs2 (installed automatically on Linux/macOS if `AUTO_INSTALL` is ticked; on Windows use WSL2, see notebook A).

**Parameters that matter.** `BUILD_DB`, `DB_NAME` (an existing folder is kept unless `OVERWRITE_EXISTING`), `SELF_TEST`, `PACKAGE_TAR`, `ARCHIVE_DIR`.

This database holds only annotated family members. It is good for clustering and for notebook C seeds, but it is **not** a database to discover new family members in: for that, search a large database (notebook A).

In [ ]:
#@title 8. Build a database { display-mode: "form" }
if COUNT_ONLY or not BUILD_DB:
    print("Database build skipped (COUNT_ONLY or BUILD_DB is not ticked).")
else:
    name = re.sub(r"\W+", "_", DB_NAME).strip("_") or "family"
    outdir = LIB / name
    if (outdir / "db.dbtype").exists() and not OVERWRITE_EXISTING:
        print(f"{outdir} already exists: keeping it (tick OVERWRITE_EXISTING to rebuild).")
        info = json.loads((outdir / "db_info.json").read_text())
    else:
        if outdir.exists(): shutil.rmtree(outdir)
        outdir.mkdir(parents=True)
        build_db_from_fasta(OUT / "proteins.fasta", outdir / "db", outdir)
        (outdir / "taxmap.tsv").unlink(missing_ok=True); shutil.rmtree(outdir / "tmp_tax", ignore_errors=True)
        info = finish_db(outdir, name, "fasta", {"family_fetch": {"interpro": IPR, "pfam": PFAM, "signatures": SIG, "reviewed_only": REVIEWED_ONLY}}, TAXIDS)
    tar = package(outdir, info)
    display(pd.DataFrame([{"name": info["name"], "sequences": f"{info['sequences']:,}", "distinct taxids": f"{info['distinct_taxids']:,}",
                           "self-test": info["checks"]["self_test"], "folder": str(outdir), "package": str(tar) if tar else ""}]))
    print("In notebooks B and C, set DATABASE to the folder (or the .tar).")

## 9. Notes: clustering this set, and very deep alignments
These are recommendations; nothing in this notebook runs them.

1. **Do not align the full-length proteins.** They have different domain compositions, so a full-length alignment of thousands of them is mostly gaps and the columns are not homologous. Use the **region files** from step 7 instead and align each domain type (lipoyl, E3-binding, catalytic) on its own. The E3-binding regions are short (tens of residues), so even tens of thousands align quickly.
2. **For many sequences, align to a profile instead of building a progressive alignment.** Build a profile HMM from a small, good seed alignment (a few hundred sequences, or the Pfam seed), then place all other sequences into it with `hmmalign` (HMMER). The cost grows linearly with the number of sequences, and the result is a consistent deep alignment. `hmmalign --trim --outformat afa` writes FASTA.
3. **If you prefer MAFFT:** for thousands of sequences use the fast settings (`--retree 2 --maxiterate 0`, or `--parttree` above roughly 10,000 sequences), and thin near-duplicates first with `mmseqs easy-cluster --min-seq-id 0.9 -c 0.8 --cov-mode 1`. FAMSA is another aligner built for very large sets (not tested here).
4. **Clustering does not need a deep alignment.** All-versus-all comparison with MMseqs2 (`mmseqs easy-cluster` at a low identity, or `mmseqs search` all-versus-all and a graph clustering) groups by sequence similarity directly. Use the alignment for trees and for checking clusters, not as the clustering input.
5. **Combine evidence.** Architecture (step 5) separates most functional groups; sequence clusters within each architecture class separate the rest. Cross-check the E2 versus E3BP boundary with notebook C, using one or more anchors per group.
6. **Look for what annotation missed.** Search the fungal reference proteomes (notebook A) with HMMs built from the region alignments, at a loose cutoff, and add the new hits to the set.